# 01 · EDA: IEEE-CIS Fraud Detection

First look at the dataset we use to train and benchmark `txn_model` (Gate A: PR-AUC > 0.80).

**Data:** `bash scripts/download_data.sh ieee` puts the files in `ml/data/raw/ieee/`.
This notebook reads `train_transaction.csv` and `train_identity.csv`.

**Sections:** 1. load · 2. shape and fraud rate · 3. missing values · 4. amount by class ·
5. fraud rate by hour · 6. top 20 features correlated with `isFraud`

In [ ]:
# Setup: imports, plot defaults and paths.
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 4)

# Repo root works whether Jupyter starts in the root or in ml/notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
IEEE_DIR = Path(os.environ.get("IEEE_DIR", ROOT / "ml" / "data" / "raw" / "ieee"))
assert (IEEE_DIR / "train_transaction.csv").exists(), (
    f"No data in {IEEE_DIR}. Run: bash scripts/download_data.sh ieee"
)

## 1. Load

In [ ]:
# Text columns, from the Kaggle data description. Everything else is numeric.
STRING_COLS = {"ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"}
STRING_COLS |= {f"M{i}" for i in range(1, 10)}
STRING_COLS |= {f"id_{i}" for i in (12, 15, 16, 23, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38)}
STRING_COLS |= {"DeviceType", "DeviceInfo"}
ID_COLS = {"TransactionID": "int64", "TransactionDT": "int64", "isFraud": "int8"}


def read_compact(path: Path) -> pd.DataFrame:
    """Read numeric columns as float32: half the memory of pandas' float64 default."""
    columns = pd.read_csv(path, nrows=0).columns
    dtypes = {c: ID_COLS.get(c, "string" if c in STRING_COLS else "float32") for c in columns}
    return pd.read_csv(path, dtype=dtypes)


txn = read_compact(IEEE_DIR / "train_transaction.csv")
idn = read_compact(IEEE_DIR / "train_identity.csv")
has_identity = txn["TransactionID"].isin(idn["TransactionID"])

# Left join: identity rows exist for only some transactions.
df = txn.merge(idn, on="TransactionID", how="left")
del txn, idn
print(f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")

## 2. Shape and fraud rate

In [ ]:
# Fraud is rare, which is why we judge models on PR-AUC and recall at 1% FPR, not accuracy.
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df["isFraud"].value_counts().rename({0: "legit", 1: "fraud"}).to_string())
print(f"Fraud rate: {df['isFraud'].mean():.2%}")
print(f"With identity info: {has_identity.mean():.1%} of transactions")
rate_with = df.loc[has_identity, "isFraud"].mean()
rate_without = df.loc[~has_identity, "isFraud"].mean()
print(f"Fraud rate with / without identity: {rate_with:.2%} / {rate_without:.2%}")

## 3. Missing values

In [ ]:
# Share missing per column. XGBoost handles NaN natively, so we keep sparse columns
# rather than impute; "is missing" itself can carry signal.
missing = df.isna().mean().sort_values(ascending=False)
bands = pd.cut(
    missing,
    bins=[-0.01, 0, 0.25, 0.5, 0.75, 0.9, 1.0],
    labels=["0%", "0-25%", "25-50%", "50-75%", "75-90%", "90-100%"],
)
print("Number of columns by missing share:")
print(bands.value_counts().sort_index().to_string())

# Top 20 most-missing columns.
missing.head(20).to_frame("missing_share").style.format("{:.1%}")

## 4. Amount by class

In [ ]:
# Summary statistics per class.
labels = {0: "legit", 1: "fraud"}
print(df.groupby("isFraud")["TransactionAmt"].describe().rename(index=labels).round(2).to_string())

# Amounts are heavily skewed, so compare shapes on a log scale (density, not counts,
# because there are far fewer fraud rows).
amount = df["TransactionAmt"].clip(lower=0.1)
bins = np.logspace(np.log10(amount.min()), np.log10(amount.max()), 60)
fig, ax = plt.subplots()
for value, name in labels.items():
    ax.hist(
        amount[df["isFraud"] == value],
        bins=bins,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=name,
    )
ax.set(
    xscale="log",
    xlabel="TransactionAmt (USD, log scale)",
    ylabel="density",
    title="Transaction amount by class",
)
ax.legend()
plt.show()

## 5. Fraud rate by hour

In [ ]:
# TransactionDT is seconds from an unknown start time, so this hour is *relative*,
# not local clock time. The daily cycle still shows up.
hour = (df["TransactionDT"] // 3600) % 24
by_hour = df.groupby(hour)["isFraud"].agg(rate="mean", n="size")

fig, ax_n = plt.subplots()
ax_n.bar(by_hour.index, by_hour["n"], color="lightgray")
ax_n.set(xlabel="hour of day (relative)", ylabel="transactions (bars)", xticks=range(0, 24, 2))
ax_rate = ax_n.twinx()
ax_rate.plot(by_hour.index, by_hour["rate"] * 100, color="crimson", marker="o")
ax_rate.set_ylabel("fraud rate % (line)")
ax_n.set_title("Volume and fraud rate by hour")
plt.show()

# Hours with the highest fraud rate.
by_hour.sort_values("rate", ascending=False).head(5).style.format({"rate": "{:.2%}", "n": "{:,}"})

## 6. Top 20 features correlated with `isFraud`

In [ ]:
# Pearson correlation of each numeric column with the label (NaN rows skipped per column).
# Linear and one feature at a time: a quick guide for feature work, not a final ranking.
numeric = df.select_dtypes("number").drop(columns=["isFraud", "TransactionID"])
corr = numeric.corrwith(df["isFraud"].astype("float32"))
top = corr.abs().sort_values(ascending=False).head(20).index

# Show missing share too: a high correlation on a 90%-empty column rests on few rows.
pd.DataFrame({"corr": corr[top], "missing_share": missing[top]}).style.format(
    {"corr": "{:+.3f}", "missing_share": "{:.1%}"}
).bar(subset=["corr"], align="mid", color=["#5b8def", "#e8665c"])

## What this means for the next tasks
- **Imbalance:** use `scale_pos_weight` and PR-AUC (T2.1), never accuracy.
- **Time:** split by `TransactionDT` (70/15/15), never randomly, so the test set is in the future.
- **Missing values:** keep NaN for tree models; consider "is missing" flags for heavy-missing groups.
- **Amount and hour:** both differ by class, which supports the amount-ratio and time features in spec 03 §2.

Write the actual numbers from this run into `docs/PROGRESS.md` (fraud rate, rows, top features).